In [4]:
import sys
sys.path.append('/host/d/Github')
import os
import numpy as np
import pandas as pd
import nibabel as nb
import matplotlib.pyplot as plt
import json
from sklearn.metrics import roc_auc_score
import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Build_lists.Build_list as Build_list

import radiomics
from radiomics import (
    featureextractor,  # This module is used for interaction with pyradiomics
)

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score

## Step 1:Train classifier on all data

### SVM

In [27]:
# ============================================================
# 0. Manually set SVM experiment variables
# ============================================================

random_state = 60
svm_feature_selector = "rfe"   # options: "rfe", "sfs", "rfecv"
top_k = 25              # used for rfe/sfs; ignored for rfecv

label_col = "Prognosis_label"

labels_path = "/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx"

whole_image_radiomics_dir = "/host/d/projects/Habitats/radiomics/whole_image"
whole_image_model_dir = "/host/d/projects/Habitats/models/whole_image/SVM"


# ============================================================
# 1. Build paths from the selected SVM setting
# ============================================================

experiment_name = f"random{random_state}_{svm_feature_selector}"
if svm_feature_selector in {"rfe", "sfs"}:
    experiment_name += f"_top{top_k}"

selected_feature_path = os.path.join(
    whole_image_radiomics_dir,
    f"radiomics_measurements_SVM_{experiment_name}_selected.xlsx",
)

best_params_path = os.path.join(
    whole_image_model_dir,
    experiment_name,
    "best_params.json",
)

print("Experiment:", experiment_name)
print("Selected feature table:", selected_feature_path)
print("Best params:", best_params_path)


# ============================================================
# 2. Read labels and selected whole-image features
# ============================================================

labels_df = pd.read_excel(labels_path)
radiomics_df = pd.read_excel(selected_feature_path)

required_label_cols = ["Patient_set", "Patient_index", label_col]
missing_label_cols = [c for c in required_label_cols if c not in labels_df.columns]
if missing_label_cols:
    raise ValueError(f"Missing columns in labels_df: {missing_label_cols}")

required_radiomics_cols = ["Patient_set", "Patient_index"]
missing_radiomics_cols = [c for c in required_radiomics_cols if c not in radiomics_df.columns]
if missing_radiomics_cols:
    raise ValueError(f"Missing columns in radiomics_df: {missing_radiomics_cols}")

merged_df = radiomics_df.merge(
    labels_df[required_label_cols],
    on=["Patient_set", "Patient_index"],
    how="inner",
    validate="one_to_one",
)

if len(merged_df) != len(radiomics_df):
    raise ValueError(
        f"Radiomics-label merge is incomplete: "
        f"radiomics={len(radiomics_df)}, merged={len(merged_df)}"
    )

non_feature_cols = ["Patient_set", "Patient_index", "Image_filepath", "Mask_filepath"]
feature_cols = [c for c in radiomics_df.columns if c not in non_feature_cols]

X_all = merged_df[feature_cols].values
y_all = merged_df[label_col].astype(int).values

print(f"Feature matrix shape: {X_all.shape}")
print(f"Label vector shape: {y_all.shape}")
print(f"Positive cases: {int(y_all.sum())}, Negative cases: {int((1 - y_all).sum())}")


# ============================================================
# 3. Read best SVM parameters
# ============================================================

with open(best_params_path, "r") as f:
    best_info = json.load(f)

best_params = best_info["best_params"]

best_C = best_params["clf__C"]
best_tol = best_params.get("clf__tol", 1e-3)

print("Loaded best SVM params:", best_params)


# ============================================================
# 4. Train final SVM model using all whole-image data
# ============================================================

svm_final = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="linear",
        C=best_C,
        tol=best_tol,
        probability=True,
        class_weight="balanced",
        random_state=random_state,
    )),
])

svm_final.fit(X_all, y_all)


# ============================================================
# 5. Apparent AUC on all training data
# ============================================================

prob_all = svm_final.predict_proba(X_all)[:, 1]
auc_all = roc_auc_score(y_all, prob_all)

print(f"Training-set apparent AUC using all data: {auc_all:.4f}")

Experiment: random60_rfe_top25
Selected feature table: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_SVM_random60_rfe_top25_selected.xlsx
Best params: /host/d/projects/Habitats/models/whole_image/SVM/random60_rfe_top25/best_params.json
Feature matrix shape: (330, 25)
Label vector shape: (330,)
Positive cases: 98, Negative cases: 232
Loaded best SVM params: {'clf__C': 10, 'clf__tol': 0.0001}
Training-set apparent AUC using all data: 0.8291


### xgboost


In [9]:
from xgboost import XGBClassifier


# ============================================================
# 0. Manually set XGBoost experiment variables
# ============================================================

random_state = 60
xgb_feature_selector = "sfs"   # options: "rfe", "sfs", "rfecv"
top_k = 15                     # used for rfe/sfs; ignored for rfecv

label_col = "Prognosis_label"

labels_path = "/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx"

whole_image_radiomics_dir = "/host/d/projects/Habitats/radiomics/whole_image"
whole_image_model_dir = "/host/d/projects/Habitats/models/whole_image/XGBoost"


# ============================================================
# 1. Build paths from the selected XGBoost setting
# ============================================================

experiment_name = f"random{random_state}_{xgb_feature_selector}"
if xgb_feature_selector in {"rfe", "sfs"}:
    experiment_name += f"_top{top_k}"

selected_feature_path = os.path.join(
    whole_image_radiomics_dir,
    f"radiomics_measurements_XGBoost_{experiment_name}_selected.xlsx",
)

best_params_path = os.path.join(
    whole_image_model_dir,
    experiment_name,
    "best_params.json",
)

print("Experiment:", experiment_name)
print("Selected feature table:", selected_feature_path)
print("Best params:", best_params_path)


# ============================================================
# 2. Read labels and selected whole-image features
# ============================================================

labels_df = pd.read_excel(labels_path)
radiomics_df = pd.read_excel(selected_feature_path)

required_label_cols = ["Patient_set", "Patient_index", label_col]
missing_label_cols = [c for c in required_label_cols if c not in labels_df.columns]
if missing_label_cols:
    raise ValueError(f"Missing columns in labels_df: {missing_label_cols}")

required_radiomics_cols = ["Patient_set", "Patient_index"]
missing_radiomics_cols = [c for c in required_radiomics_cols if c not in radiomics_df.columns]
if missing_radiomics_cols:
    raise ValueError(f"Missing columns in radiomics_df: {missing_radiomics_cols}")

merged_df = radiomics_df.merge(
    labels_df[required_label_cols],
    on=["Patient_set", "Patient_index"],
    how="inner",
    validate="one_to_one",
)

if len(merged_df) != len(radiomics_df):
    raise ValueError(
        f"Radiomics-label merge is incomplete: "
        f"radiomics={len(radiomics_df)}, merged={len(merged_df)}"
    )

non_feature_cols = ["Patient_set", "Patient_index", "Image_filepath", "Mask_filepath"]
feature_cols = [c for c in radiomics_df.columns if c not in non_feature_cols]

X_all = merged_df[feature_cols].values
y_all = merged_df[label_col].astype(int).values

print(f"Feature matrix shape: {X_all.shape}")
print(f"Label vector shape: {y_all.shape}")
print(f"Positive cases: {int(y_all.sum())}, Negative cases: {int((1 - y_all).sum())}")


# ============================================================
# 3. Read best XGBoost parameters
# ============================================================

with open(best_params_path, "r") as f:
    best_info = json.load(f)

best_params = best_info["best_params"]

scale_pos_weight = best_info.get(
    "scale_pos_weight",
    float(np.sum(y_all == 0) / np.sum(y_all == 1))
)

print("Loaded best XGBoost params:", best_params)
print("scale_pos_weight:", scale_pos_weight)


# ============================================================
# 4. Train final XGBoost model using all whole-image data
# ============================================================

xgb_final = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=random_state,
    n_jobs=1,
    scale_pos_weight=scale_pos_weight,

    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    learning_rate=best_params["learning_rate"],
)

xgb_final.fit(X_all, y_all)


# ============================================================
# 5. Apparent AUC on all training data
# ============================================================

prob_all = xgb_final.predict_proba(X_all)[:, 1]
auc_all = roc_auc_score(y_all, prob_all)

print(f"Training-set apparent AUC using all data: {auc_all:.4f}")

Experiment: random60_sfs_top15
Selected feature table: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_XGBoost_random60_sfs_top15_selected.xlsx
Best params: /host/d/projects/Habitats/models/whole_image/XGBoost/random60_sfs_top15/best_params.json
Feature matrix shape: (330, 15)
Label vector shape: (330,)
Positive cases: 98, Negative cases: 232
Loaded best XGBoost params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 100}
scale_pos_weight: 2.36734693877551
Training-set apparent AUC using all data: 1.0000


### KNN

In [10]:
from sklearn.neighbors import KNeighborsClassifier
random_state = 60
knn_feature_selector = "sfs"   # current option: "sfs"
top_k = 15

label_col = "Prognosis_label"

labels_path = "/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx"

whole_image_radiomics_dir = "/host/d/projects/Habitats/radiomics/whole_image"
whole_image_model_dir = "/host/d/projects/Habitats/models/whole_image/KNN"


# ============================================================
# 1. Build paths from the selected KNN setting
# ============================================================

experiment_name = f"random{random_state}_{knn_feature_selector}_top{top_k}"

selected_feature_path = os.path.join(
    whole_image_radiomics_dir,
    f"radiomics_measurements_KNN_{experiment_name}_selected.xlsx",
)

best_params_path = os.path.join(
    whole_image_model_dir,
    experiment_name,
    "best_params.json",
)

print("Experiment:", experiment_name)
print("Selected feature table:", selected_feature_path)
print("Best params:", best_params_path)


# ============================================================
# 2. Read labels and selected whole-image features
# ============================================================

labels_df = pd.read_excel(labels_path)
radiomics_df = pd.read_excel(selected_feature_path)

required_label_cols = ["Patient_set", "Patient_index", label_col]
missing_label_cols = [c for c in required_label_cols if c not in labels_df.columns]
if missing_label_cols:
    raise ValueError(f"Missing columns in labels_df: {missing_label_cols}")

required_radiomics_cols = ["Patient_set", "Patient_index"]
missing_radiomics_cols = [c for c in required_radiomics_cols if c not in radiomics_df.columns]
if missing_radiomics_cols:
    raise ValueError(f"Missing columns in radiomics_df: {missing_radiomics_cols}")

merged_df = radiomics_df.merge(
    labels_df[required_label_cols],
    on=["Patient_set", "Patient_index"],
    how="inner",
    validate="one_to_one",
)

if len(merged_df) != len(radiomics_df):
    raise ValueError(
        f"Radiomics-label merge is incomplete: "
        f"radiomics={len(radiomics_df)}, merged={len(merged_df)}"
    )

non_feature_cols = ["Patient_set", "Patient_index", "Image_filepath", "Mask_filepath"]
feature_cols = [c for c in radiomics_df.columns if c not in non_feature_cols]

X_all = merged_df[feature_cols].values
y_all = merged_df[label_col].astype(int).values

print(f"Feature matrix shape: {X_all.shape}")
print(f"Label vector shape: {y_all.shape}")
print(f"Positive cases: {int(y_all.sum())}, Negative cases: {int((1 - y_all).sum())}")


# ============================================================
# 3. Read best KNN parameters
# ============================================================

with open(best_params_path, "r") as f:
    best_info = json.load(f)

best_params = best_info["best_params"]

print("Loaded best KNN params:", best_params)


# ============================================================
# 4. Train final KNN model using all whole-image data
# ============================================================

knn_final = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier(
        n_neighbors=best_params["clf__n_neighbors"],
        weights=best_params["clf__weights"],
    )),
])

knn_final.fit(X_all, y_all)


# ============================================================
# 5. Apparent AUC on all training data
# ============================================================

prob_all = knn_final.predict_proba(X_all)[:, 1]
auc_all = roc_auc_score(y_all, prob_all)

print(f"Training-set apparent AUC using all data: {auc_all:.4f}")

Experiment: random60_sfs_top15
Selected feature table: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_KNN_random60_sfs_top15_selected.xlsx
Best params: /host/d/projects/Habitats/models/whole_image/KNN/random60_sfs_top15/best_params.json
Feature matrix shape: (330, 15)
Label vector shape: (330,)
Positive cases: 98, Negative cases: 232
Loaded best KNN params: {'clf__n_neighbors': 5, 'clf__weights': 'distance'}
Training-set apparent AUC using all data: 1.0000


## Step 2: Run the model on each case

In [ ]:
# ============================================================
# 0. Manually set SVM experiment variables
# ============================================================

random_state = 60
svm_feature_selector = "rfe"   # options: "rfe", "sfs", "rfecv"
top_k = 25                    # used for rfe/sfs; ignored for rfecv

clip_to_01 = True
skip_if_done = False


whole_image_radiomics_dir = "/host/d/projects/Habitats/radiomics/whole_image"
habitat_radiomics_root = "/host/d/projects/Habitats/radiomics/habitats"

labels_path = "/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx"


# ============================================================
# 1. Build SVM experiment name and selected feature path
# ============================================================

experiment_name = f"random{random_state}_{svm_feature_selector}"
if svm_feature_selector in {"rfe", "sfs"}:
    experiment_name += f"_top{top_k}"

selected_feature_path = os.path.join(
    whole_image_radiomics_dir,
    f"radiomics_measurements_SVM_{experiment_name}_selected.xlsx",
)

print("SVM experiment:", experiment_name)
print("Selected feature table:", selected_feature_path)

if not os.path.isfile(selected_feature_path):
    raise FileNotFoundError(f"Selected feature table not found: {selected_feature_path}")

selected_feature_df = pd.read_excel(selected_feature_path)

non_feature_cols_whole = [
    "Patient_set",
    "Patient_index",
    "Image_filepath",
    "Mask_filepath",
]

selected_feature_cols = [
    c for c in selected_feature_df.columns
    if c not in non_feature_cols_whole
]

# ============================================================
# 1A. Safety check: terminate if original shape features are selected
# ============================================================

selected_shape_features = [
    f for f in selected_feature_cols
    if f.startswith("original_shape_")
]

if selected_shape_features:
    print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    print("WARNING: original shape features were selected.")
    print("These features are not suitable for patch-level habitat projection,")
    print("because patch shape mainly reflects patch geometry/mask geometry,")
    print("not whole-tumor biological morphology.")
    print("\nSelected original shape features:")
    for f in selected_shape_features:
        print("  ", f)
    print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n")

    raise RuntimeError(
        "Terminated because selected features contain original_shape_* features. "
        "Please choose another whole-image model/feature-selection setting."
    )


# ============================================================
# 2. Build patient list
# ============================================================

build = Build_list.Build(labels_path)
batch_list, patient_set_list, patient_index_list, label_list, image_path_list, mask_path_list = build.__build__()

print(f"Number of cases to process: {len(image_path_list)}")



# ============================================================
# 3. Run final SVM model on each patient's patches
# ============================================================

for i in range(0,1):
    patient_set = patient_set_list[i]
    patient_index = patient_index_list[i]
    print("\n============================================================")
    print("Processing patient set:", patient_set, "index:", patient_index, ' i =', i)

    patient_dir = os.path.join(habitat_radiomics_root, str(patient_set), str(patient_index))

    radiomics_path = os.path.join(
        patient_dir,
        "radiomics_features_patches_normalized.xlsx",
    )

    output_path = os.path.join(
        patient_dir,
        f"radiomics_features_patches_prob_clusters_SVM_{experiment_name}.xlsx",
    )

    qc_path = os.path.join(
        patient_dir,
        f"patch_feature_range_QC_SVM_{experiment_name}.xlsx",
    )

    if skip_if_done and os.path.isfile(output_path):
        print(f"Skipping patient index {patient_index} as output file already exists.")
        continue

    if not os.path.isfile(radiomics_path):
        raise FileNotFoundError(f"Patch radiomics file not found: {radiomics_path}")

    radiomics_df = pd.read_excel(radiomics_path)

    missing_features = [
        f for f in selected_feature_cols
        if f not in radiomics_df.columns
    ]
    if missing_features:
        raise ValueError(
            f"Patient {patient_index} is missing selected features: {missing_features}"
        )

    X_patch_raw = radiomics_df[selected_feature_cols].astype(float)

    # ========================================================
    # 3A. QC before clipping: out-of-range report per feature
    # ========================================================

    qc_rows = []
    n_patches = len(X_patch_raw)

    for feature in selected_feature_cols:
        values = X_patch_raw[feature].values

        n_below_0 = int(np.sum(values < 0))
        n_above_1 = int(np.sum(values > 1))

        qc_rows.append({
            "Patient_index": patient_index,
            "feature": feature,
            "n_patches": n_patches,
            "n_below_0": n_below_0,
            "pct_below_0": 100 * n_below_0 / n_patches if n_patches > 0 else np.nan,
            "n_above_1": n_above_1,
            "pct_above_1": 100 * n_above_1 / n_patches if n_patches > 0 else np.nan,
            "min": float(np.nanmin(values)),
            "max": float(np.nanmax(values)),
            "mean": float(np.nanmean(values)),
            "median": float(np.nanmedian(values)),
        })

    qc_df = pd.DataFrame(qc_rows)
    qc_df.to_excel(qc_path, index=False)

    total_values = X_patch_raw.size
    total_below_0 = int((X_patch_raw < 0).sum().sum())
    total_above_1 = int((X_patch_raw > 1).sum().sum())

    print("Patch feature range QC:")
    print(f"  total values: {total_values}")
    print(
        f"  below 0: {total_below_0} "
        f"({100 * total_below_0 / total_values:.2f}%)"
    )
    print(
        f"  above 1: {total_above_1} "
        f"({100 * total_above_1 / total_values:.2f}%)"
    )
    print("Saved QC:", qc_path)

    print("\nTop features with values below 0:")
    print(
        qc_df.sort_values("pct_below_0", ascending=False)
        [["feature", "pct_below_0", "min", "max"]]
        .head(10)
        .to_string(index=False)
    )

    print("\nTop features with values above 1:")
    print(
        qc_df.sort_values("pct_above_1", ascending=False)
        [["feature", "pct_above_1", "min", "max"]]
        .head(10)
        .to_string(index=False)
    )

    # ========================================================
    # 3B. Clip to [0, 1], then predict patch probability
    # ========================================================

    X_patch = X_patch_raw.values

    if clip_to_01:
        X_patch = np.clip(X_patch, 0, 1)

    prob = svm_final.predict_proba(X_patch)[:, 1]
    print(f"Probability vector shape: {prob.shape}")
    print(f"Probability range: {prob.min():.4f} - {prob.max():.4f}")

    # ========================================================
    # 3C. Cluster patch probabilities into low/high habitats
    # ========================================================

    p = prob.reshape(-1, 1)

    K = 2
    kmeans = KMeans(n_clusters=K, random_state=0, n_init="auto")
    cluster_id = kmeans.fit_predict(p)

    print("cluster_id shape:", cluster_id.shape)
    print("cluster counts before adjustment:", np.bincount(cluster_id))

    cluster_means = np.array([
        prob[cluster_id == c].mean()
        for c in range(K)
    ])

    # Ensure cluster 1 is the high-probability habitat.
    if cluster_means[0] > cluster_means[1]:
        cluster_id = 1 - cluster_id

    cluster_means = np.array([
        prob[cluster_id == c].mean()
        for c in range(K)
    ])

    print("cluster mean prob after adjustment:", cluster_means)
    print("cluster counts after adjustment:", np.bincount(cluster_id))

    # ========================================================
    # 3D. Save selected patch features, probability and habitat cluster
    # ========================================================

    metadata_cols = [
        "Patient_set",
        "Patient_index",
        "patch_id",
        "tumor_fraction",
        "Image_filepath",
        "Mask_filepath",
    ]

    metadata_cols = [c for c in metadata_cols if c in radiomics_df.columns]

    save_df = radiomics_df[metadata_cols + selected_feature_cols].copy()

    save_df["prob_SVM"] = prob
    save_df["cluster_SVM"] = cluster_id
    save_df["SVM_experiment"] = experiment_name
    save_df["clip_to_01"] = clip_to_01

    save_df.to_excel(output_path, index=False)
    print("Saved selected patch features, probability and clusters:", output_path)

SVM experiment: random60_rfe_top25
Selected feature table: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_SVM_random60_rfe_top25_selected.xlsx
Number of cases to process: 330

Processing patient set: set_1 index: 1  i = 0
Patch feature range QC:
  total values: 14900
  below 0: 6203 (41.63%)
  above 1: 2775 (18.62%)
Saved QC: /host/d/projects/Habitats/radiomics/habitats/set_1/1/patch_feature_range_QC_SVM_random60_rfe_top25.xlsx

Top features with values below 0:
                                                      feature  pct_below_0        min       max
        wavelet-HHL_gldm_LargeDependenceHighGrayLevelEmphasis   100.000000  -0.055831 -0.048270
                      wavelet-HHL_glszm_SizeZoneNonUniformity   100.000000  -0.002426 -0.000807
             log-sigma-2-0-mm-3D_glszm_GrayLevelNonUniformity   100.000000  -0.022077 -0.012097
log-sigma-2-0-mm-3D_gldm_LargeDependenceHighGrayLevelEmphasis   100.000000  -0.092821 -0.064416
                             

### turn ROI into habitats

In [7]:
### do it for each case - each patch

build = Build_list.Build(os.path.join('/host/d/Data/Habitats/Jishuitan/Patient_lists', 'labels_with_image_info_included.xlsx'))
batch_list, patient_index_list, label_list, image_path_list, mask_path_list = build.__build__()
print(f'Number of cases to process: {len(image_path_list)}')

for i in range(0, len(patient_index_list)):
    patient_index = patient_index_list[i]
    print('Processing patient index:', patient_index, ' i is ', i)

    # if done skip
    # if os.path.isfile( os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_index), 'ROI_low.nii.gz')) == 1:
    #     print('done, continue')
    #     continue
    
    radiomics_path = os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_index), 'radiomics_features_patches_prob_clusters.xlsx')

    # load a template patch
    template_file = os.path.join('/host/d/Data/Habitats/Jishuitan/original_data/', str(patient_index), 'img.nii.gz')
    template_patch = nb.load(template_file).get_fdata()
    affine = nb.load(template_file).affine
    header = nb.load(template_file).header
    # turn it into all zeros
    ROI_high = np.zeros_like(template_patch)
    ROI_low = np.zeros_like(template_patch)


    cluster_info = pd.read_excel(radiomics_path)
    # find all the rows with cluster
    high_cluster_rows = cluster_info[cluster_info['cluster'] == 1]
    low_cluster_rows = cluster_info[cluster_info['cluster'] == 0]

    for j in range(0,high_cluster_rows.shape[0]):
        patch_id = high_cluster_rows.iloc[j]['patch_id']
        patch_file = os.path.join('/host/d/Data/Habitats/Jishuitan/habitats/', str(patient_index), f'patches/patch_{patch_id:04d}.nii.gz')
        patch_data = nb.load(patch_file).get_fdata()
        ROI_high += patch_data
    
    # assert ROI high only has 0 or 1 value
    assert np.all((ROI_high == 0) | (ROI_high == 1))

    for j in range(0,low_cluster_rows.shape[0]):
        patch_id = low_cluster_rows.iloc[j]['patch_id']
        patch_file = os.path.join('/host/d/Data/Habitats/Jishuitan/habitats/', str(patient_index), f'patches/patch_{patch_id:04d}.nii.gz')
        patch_data = nb.load(patch_file).get_fdata()
        ROI_low += patch_data
    assert np.all((ROI_low == 0) | (ROI_low == 1))


    # save 
    nb.save(nb.Nifti1Image(ROI_high, affine,header), os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_index), 'ROI_high.nii.gz'))
    nb.save(nb.Nifti1Image(ROI_low, affine,header), os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_index), 'ROI_low.nii.gz'))


Number of cases to process: 81
Processing patient index: 5  i is  0
Processing patient index: 7  i is  1
Processing patient index: 8  i is  2
Processing patient index: 11  i is  3
Processing patient index: 15  i is  4
Processing patient index: 18  i is  5
Processing patient index: 19  i is  6
Processing patient index: 20  i is  7
Processing patient index: 21  i is  8
Processing patient index: 22  i is  9
Processing patient index: 23  i is  10
Processing patient index: 24  i is  11
Processing patient index: 26  i is  12
Processing patient index: 28  i is  13
Processing patient index: 29  i is  14
Processing patient index: 30  i is  15
Processing patient index: 33  i is  16
Processing patient index: 34  i is  17
Processing patient index: 36  i is  18
Processing patient index: 38  i is  19
Processing patient index: 41  i is  20
Processing patient index: 42  i is  21
Processing patient index: 43  i is  22
Processing patient index: 44  i is  23
Processing patient index: 46  i is  24
Process